# Aliado Libre — fine-tuning Qwen3.5-2B en GPU (Google Colab)

Este notebook entrena SOLO **Qwen3.5-2B**. No hay nada que configurar ni elegir — solo correr todo y subir el dataset cuando lo pida.

**Antes de correr:** `Entorno de ejecución` → `Cambiar tipo de entorno de ejecución` → GPU (T4 alcanza).

**Pasos:**
1. `Entorno de ejecución` → `Ejecutar todas`.
2. Cuando la celda de carga pida el archivo, sube `finetune/data/entrenamiento.jsonl` (el dataset nuevo de 185 ejemplos).
3. Al final se descarga `modelo_lora_qwen35_2b.zip`.
4. Descomprime ese zip dentro de `finetune/` en tu proyecto local, en la carpeta `finetune/modelo_lora_qwen35_2b/`.

¿Quieres entrenar también el 0.8B? Usa el notebook separado `colab_entrenar_qwen35_08b.ipynb` en una sesión de Colab **nueva** (no reuses esta pestaña/sesión para el otro modelo).

In [1]:
!pip install -q -U transformers peft trl accelerate datasets

## Configuración (fija — este notebook es solo para Qwen3.5-2B)

In [2]:
# No edites esto — este notebook siempre entrena Qwen3.5-2B.
MODELO_BASE = "Qwen/Qwen3.5-2B"
NOMBRE_SALIDA = "modelo_lora_qwen35_2b"
NUM_EPOCHS = 3

## Subir el dataset
Sube `finetune/data/entrenamiento.jsonl` (50 ejemplos, ya preparado en el proyecto local).

In [ ]:
from google.colab import files
subido = files.upload()
RUTA_DATASET = list(subido.keys())[0]
print(f"Dataset cargado: {RUTA_DATASET}")

In [ ]:
import json
import torch
from datasets import Dataset
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import SFTConfig, SFTTrainer

assert torch.cuda.is_available(), "No hay GPU activa — revisa Entorno de ejecución > Cambiar tipo de entorno de ejecución"
print("GPU:", torch.cuda.get_device_name(0))

ejemplos = [json.loads(l) for l in open(RUTA_DATASET, encoding="utf-8") if l.strip()]
textos = [f"{e['prompt']}{e['completion']}" for e in ejemplos]
dataset = Dataset.from_dict({"text": textos})
print(f"{len(dataset)} ejemplos cargados")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODELO_BASE)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

modelo = AutoModelForCausalLM.from_pretrained(MODELO_BASE, dtype="bfloat16").cuda()

# Verificación dura: si por cualquier motivo (sesión no reiniciada entre corridas,
# caché de Colab, etc.) el modelo cargado NO es el que pediste en MODELO_BASE,
# esto detiene todo con un error claro en vez de dejarte guardar y descargar
# silenciosamente el modelo equivocado bajo el nombre del otro experimento.
# Se compara contra el hidden_size que reporta el config.json del propio
# repo de HF (no una tabla fija) para que funcione con cualquier MODELO_BASE.
from transformers import AutoConfig

_config_esperado = AutoConfig.from_pretrained(MODELO_BASE)
_hidden_size_esperado = getattr(_config_esperado, "hidden_size", None)
print(f"Modelo cargado: {MODELO_BASE} | hidden_size real = {modelo.config.hidden_size}")
if _hidden_size_esperado is not None:
    assert modelo.config.hidden_size == _hidden_size_esperado, (
        f"¡Se pidió {MODELO_BASE} (hidden_size={_hidden_size_esperado} según su config.json) pero se "
        f"cargó un modelo con hidden_size={modelo.config.hidden_size}! No sigas — reinicia el entorno "
        f"de ejecución completo (Entorno de ejecución > Desconectar y eliminar tiempo de ejecución) y "
        f"vuelve a correr todo desde cero."
    )

config_lora = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type="CAUSAL_LM",
)
modelo = get_peft_model(modelo, config_lora)
modelo.print_trainable_parameters()

In [ ]:
argumentos = SFTConfig(
    output_dir="checkpoints",
    bf16=True,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    logging_steps=1,
    save_strategy="epoch",
    report_to=[],
    max_length=1024,
    dataset_text_field="text",
)

entrenador = SFTTrainer(
    model=modelo,
    args=argumentos,
    train_dataset=dataset,
    processing_class=tokenizer,
)

entrenador.train()

In [ ]:
modelo.save_pretrained(NOMBRE_SALIDA)
tokenizer.save_pretrained(NOMBRE_SALIDA)
print(f"Adaptador guardado en {NOMBRE_SALIDA}/")

## Descargar el resultado
Comprime el adaptador y lo descarga — descomprímelo en `finetune/` dentro de tu proyecto local (reemplazando la carpeta `finetune/{NOMBRE_SALIDA}/`).

In [ ]:
import shutil
shutil.make_archive(NOMBRE_SALIDA, "zip", NOMBRE_SALIDA)
files.download(f"{NOMBRE_SALIDA}.zip")